# Python Notebook that is used to train and export the model for the ESP32

Install dependencies, ideally in a dedicated environment (.venv)

In [3]:
%pip install -r requirements.txt

  Using cached absl_py-2.4.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
  Using cached charset_normalizer-3.4.6-cp313-cp313-win_amd64.whl.metadata (41 kB)
  Using cached contourpy-1.3.3-cp313-cp313-win_amd64.whl.metadata (5.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached grpcio-1.78.0-cp313-cp313-win_amd64.whl.metadata (3.9 kB)
  Using cached h5py-3.14.0-cp313-cp313-win_amd64.whl.metadata (2.7 kB)
  Using cached keras-3.13.2-py3-none-any.whl.metadata (6.3 kB)
  Using cached markdown_it_py-4.0.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached matplotlib-3.10.8-cp313-cp313-win_amd64.whl.metadata (52 kB)
  Using cached ml_dtypes-0.5.4-cp313-cp313-win_amd64.whl.metadata (9.2 kB)
  Using cached optree-0.19.0-cp313-cp313-win_amd64.whl.metadata (35 kB)
  Using cached pandas-3.0.1-cp313-cp313-win_amd64.whl.metadata (19

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow import lite as tfl

from keras import Model, Input
from keras.models import Sequential
from keras.layers import Conv1D, Dense, Dropout, AveragePooling1D, Flatten, LSTM

from matplotlib import pyplot as plt

from sklearn.metrics import accuracy_score

In [38]:
# prepare data
X_train = None
y_train = None
X_test = None
y_test = None

In [9]:
# change these to what you need for the input shape

samples_per_classification = 100 # so a second of data
signal_count = 6 # the six axes of the IMU

In [10]:
# inputs and outputs of the model
input_shape = (samples_per_classification, signal_count)
num_classes = 7

In [28]:
# this is the LSTM version, may not fit on the board
def lstm_model(input_shape=input_shape, num_classes=num_classes, cells_first=64, cells_second=32) -> Model:
    lstm = Sequential()
    lstm.add(Input(shape=input_shape))
    lstm.add(LSTM(cells_first, return_sequences=True))
    lstm.add(Dropout(0.2))
    lstm.add(LSTM(cells_second))
    lstm.add(Dropout(0.2))
    lstm.add(Dense(cells_second, activation="relu"))
    lstm.add(Dense(num_classes, activation="softmax"))
    
    return lstm

In [23]:
# This is the CNN model
def cnn_model(input_shape=input_shape, num_classes=num_classes, filters=8, kernel_size=9) -> Model:
    cnn = Sequential()
    cnn.add(Input(shape=input_shape))
    cnn.add(Conv1D(filters=filters, kernel_size=kernel_size))
    cnn.add(Dropout(rate=0.2))
    cnn.add(AveragePooling1D(pool_size=5))
    cnn.add(Flatten())
    cnn.add(Dense(100, activation="relu"))
    cnn.add(Dropout(0.1))
    cnn.add(Dense(num_classes, activation="softmax"))

    return cnn

In [41]:
model = lstm_model()
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy', 'val_accuracy'])

model.summary()

Model: "sequential_14"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_16 (LSTM)                  │ (None, 100, 64)        │        18,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_22 (Dropout)            │ (None, 100, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_17 (LSTM)                  │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_23 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 7)              │           231 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31,879 (124.53 KB)

 Trainable params: 31,879 (124.53 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# train the model
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2)

plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper left')
plt.show()

In [ ]:
# test the model
predictions = model.predict(X_test)  # Shape: (samples, num_classes)
y_pred = tf.argmax(predictions, axis=1)

accuracy = 100 * accuracy_score(y_test, y_pred)
print(f"Accuracy of model: {accuracy}%")

In [ ]:
# save the model

# change the file name if needed
model_location = 'model.h5'
model.save(model_location)

# change the converted file name if needed
converted_location = 'model.tflite'

# change the C header file name if needed
# this is what is deployed to the ESP32
header_location = "model.h"

Convert the model to tflite file

In [ ]:
# run this cell if you want to also quantize the model
# otherwise just run the command below

converter = tfl.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tfl.Optimize.DEFAULT]
converter.representative_dataset = X_test
converter.target_spec.supported_ops = [tfl.OpsSet.TFLITE_BUILTINS_INT8]
tflite_quant_model = converter.convert()
with open(converted_location, 'wb') as f:
    f.write(tflite_quant_model)

In [ ]:
# !tflite_convert --keras_model_file {model_location} --output_file {converted location}

Convert tflite file to C header; make sure xxd is available in the system PATH

In [ ]:
!xxd -i {converted_location} > {header_location}